<a href="https://colab.research.google.com/github/leespace1231/leespace1231.github.io/blob/main/%EA%B3%BC%EC%A0%9C6(4/17).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import gradio as gr
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 데이터 준비 (최초 1회 실행)
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                               remove=('headers', 'footers', 'quotes'))

data, labels = [], []
for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20] # 주제당 20개
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

# 2. 벡터화 모델 학습
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

# 3. 분류 핵심 함수
def predict_category(text):
    if not text.strip():
        return "텍스트를 입력해주세요.", 0.0

    # 입력 문장 벡터화 및 유사도 계산
    input_vec = vectorizer.transform([text])
    sim = cosine_similarity(input_vec, count_matrix)

    best_idx = np.argmax(sim)
    max_sim = sim[0][best_idx]

    if max_sim == 0:
        return "분류 불가 (학습된 단어 없음)", 0.0

    return labels[best_idx], round(float(max_sim), 4)

# 4. Gradio 인터페이스 구성
demo = gr.Interface(
    fn=predict_category,
    inputs=gr.Textbox(lines=5, placeholder="분류할 뉴스 텍스트를 영어로 입력하세요...", label="입력 텍스트"),
    outputs=[
        gr.Label(label="예측된 주제"),
        gr.Number(label="코사인 유사도 점수")
    ],
    title="📰 AI 뉴스 그룹 주제 분류기",
    description="텍스트를 입력하면 가장 유사한 뉴스 카테고리(그래픽, 우주, 종교)를 찾아줍니다.",
    examples=[
        ["The telescope captured a beautiful image of the nebula."],
        ["What is the nature of soul and divine power?"],
        ["How to render 3D objects using Python?"]
    ]
)

# 5. 실행 (share=True로 외부 접속 링크 생성)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://814cb13949493cd99b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
